# L2 · Extracción con Docling y cascada de seccionado

Docling real (`do_ocr=False`, 1 worker, subproceso con timeout, caché SHA-256) y la cascada de docs/07 §2: patrones → tipografía → longitud.
Se ejecuta con `taskset -c 0,1` y `OMP_NUM_THREADS=2` (los núcleos 2 y 3 son de otros desarrolladores). **Criterio (docs/13):** los artículos de la norma LA/FT coinciden con el conteo manual (≥ 95 %) y reprocesar el mismo PDF usa la caché.

In [ ]:
# --- Arranque por stage (ver 00_modelos.ipynb) ---------------------------------------------------
# Local: el stage sale de config/.env (STAGE=dev MacBook+DMR · STAGE=sandbox Raspberry+Groq+Ollama). Nada que hacer.
# Databricks (STAGE=mvp): descomenta y ejecuta ANTES de importar `backend`.
# %pip install -r ../requirements.txt
# dbutils.library.restartPython()
# import os
# os.environ["STAGE"] = "mvp"
# os.environ["FOUNDRY_AI_TOKEN"] = dbutils.secrets.get(scope="comparador", key="foundry-token")


In [1]:
import os, re, sys, time, json, resource, tempfile
from pathlib import Path
from collections import Counter

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pymupdf
from backend.config import settings as s
from backend.extraction.docling_extractor import extraer, limpiar_cache, ExtraccionError
from backend.extraction.sectioner import parsear_bloques, negritas_pdf, SeccionadoError

NORMAS = Path(os.environ.get("NORMAS_DIR", RAIZ.parent.parent / "comparador-normativas-ec-v1" / "Normativa2026"))
CAP3 = NORMAS / "L1-XVI-cap-III.pdf"   # norma corta de la SB (3 págs)
LAFT = NORMAS / "Proyecto-de-Ley-Organica-Organica-para-Reprimir-y-Prevenir-el-Lavado-de-Activos-y-la-Financiacion-del-Terrorismo.pdf"
MOCKS = sorted((RAIZ / "samples").glob("MOCK-DEMO-0*.pdf"))

print("núcleos permitidos:", sorted(os.sched_getaffinity(0)), "· OMP_NUM_THREADS =", os.environ.get("OMP_NUM_THREADS"))
print("Docling:", f"tablas={s.DOCLING_TABLES}", f"hilos={s.DOCLING_THREADS}", f"timeout={s.DOCLING_TIMEOUT_S}s", f"tanda={s.DOCLING_CHUNK_PAGES} págs")
print("Seccionado:", f"SECCION_CHARS={s.SECCION_CHARS}", f"SECCION_BLOQUE_MAX={s.SECCION_BLOQUE_MAX}")
print("LA/FT:", LAFT.name, "·", pymupdf.open(LAFT).page_count, "páginas")

def ram_hijos_mb():
    # Pico de RAM del subproceso de Docling (el mayor de los hijos ya terminados).
    return resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1024

núcleos permitidos: [0, 1] · OMP_NUM_THREADS = 2
Docling: tablas=fast hilos=2 timeout=1500s tanda=10 págs
Seccionado: SECCION_CHARS=1200 SECCION_BLOQUE_MAX=200000
LA/FT: Proyecto-de-Ley-Organica-Organica-para-Reprimir-y-Prevenir-el-Lavado-de-Activos-y-la-Financiacion-del-Terrorismo.pdf · 60 páginas


## 1 · Extracción real de la norma LA/FT, con progreso y medición

In [2]:
limpiar_cache()
avance = []
t0 = time.perf_counter()
bloques = extraer(LAFT, lambda hechas, total: avance.append((hechas, total, round(time.perf_counter() - t0))))
t_extraccion = time.perf_counter() - t0
ram_pico = ram_hijos_mb()

tipos = Counter(b["tipo"] for b in bloques)
niveles = Counter(b["nivel"] for b in bloques if b["tipo"] == "encabezado")
print(f"{len(bloques)} bloques en {t_extraccion:.1f} s · RAM pico del subproceso {ram_pico:.0f} MB")
print("progreso (páginas hechas, total, segundos):", avance)
print("tipos:", dict(tipos), "· niveles de encabezado:", dict(sorted(niveles.items())))
for b in bloques[:5] + [x for x in bloques if x["tipo"] in ("tabla", "lista")][:3]:
    print(f"  p.{b['pagina']} [{b['tipo']}{' n'+str(b['nivel']) if 'nivel' in b else ''}] {b['texto'][:70]!r}")

591 bloques en 627.5 s · RAM pico del subproceso 2430 MB
progreso (páginas hechas, total, segundos): [(10, 60, 109), (20, 60, 193), (30, 60, 271), (40, 60, 332), (50, 60, 387), (60, 60, 626)]
tipos: {'encabezado': 88, 'parrafo': 143, 'lista': 352, 'tabla': 8} · niveles de encabezado: {1: 88}
  p.1 [encabezado n1] 'Memorando Nro. AN-PR-2022-0559-M'
  p.1 [encabezado n1] 'Quito, D.M., 12 de octubre de 2022'
  p.1 [parrafo] 'PARA:'
  p.1 [parrafo] 'Sr. Abg. Álvaro Ricardo Salazar Paredes Secretario General'
  p.1 [parrafo] 'ASUNTO: Difusión del Proyecto de Ley Orgánica para Reprimir y Prevenir'
  p.1 [lista] 'AN-ZOEJ-2022-0034-M'
  p.1 [lista] 'vf-oficio_asambleistas_-signed_firmado.pdf'
  p.1 [lista] 'ficha_de_verificacioÌn_de_cumplimiento_ods_(1)0875323001665261475.pdf'


## 2 · Caché por SHA-256: el mismo PDF no vuelve a pasar por Docling

In [3]:
t0 = time.perf_counter()
otra_vez = extraer(LAFT)
t_cache = time.perf_counter() - t0
print(f"segunda llamada: {t_cache:.3f} s (primera: {t_extraccion:.1f} s) · mismo resultado: {otra_vez == bloques}")

segunda llamada: 0.242 s (primera: 627.5 s) · mismo resultado: True


## 3 · Los PDFs no válidos se rechazan antes de llegar a Docling

In [4]:
falso = Path(tempfile.mkdtemp()) / "falso.pdf"; falso.write_text("no es un pdf")
try:
    extraer(falso)
    rechazo = None
except ExtraccionError as e:
    rechazo = e.codigo
    print(e.codigo, "·", e.para_usuario()["mensaje"])

ERR-ING-003 · No pudimos abrir este archivo. Verifica que sea un PDF válido y sin contraseña.


## 4 · Seccionado de la LA/FT (cascada) contra el conteo manual
**Conteo manual:** regex independiente sobre el texto de PyMuPDF (no sobre Docling): `Artículo N.-` al inicio de línea. Las líneas que empiezan por "art" y no cumplen esa regex se listan como **casos dudosos** y se revisan a mano.

In [5]:
avisos = []
t0 = time.perf_counter()
secciones = parsear_bloques(bloques, doc_id="N1", tipo_doc="normativa", pdf=LAFT, avisos=avisos)
t_seccionado = time.perf_counter() - t0
print(f"{len(secciones)} secciones en {t_seccionado:.3f} s · estrategias: {dict(Counter(x.estrategia for x in secciones))}")
print("niveles:", dict(Counter(x.nivel for x in secciones)), "· hojas:", sum(x.es_hoja for x in secciones))
print("avisos:", avisos)

# Conteo manual (independiente del seccionador)
texto_pdf = "\n".join(p.get_text() for p in pymupdf.open(LAFT))
estricta = re.compile(r"^[ \t]*Art[ií]culo[ \t]+(\d+)[ \t]*\.?[ \t]*-", re.I | re.M)
esperados = [int(n) for n in estricta.findall(texto_pdf)]
dudosos = [l.strip() for l in texto_pdf.splitlines() if re.match(r"[ \t]*art", l, re.I) and not estricta.match(l)]
print(f"\nesperados (regex independiente): {len(esperados)} · rango {min(esperados)}–{max(esperados)} · sin saltos ni repetidos: {sorted(esperados) == list(range(1, max(esperados)+1))}")
print("casos dudosos revisados a mano:")
for l in dudosos:
    print("  ", l[:90])

143 secciones en 0.552 s · estrategias: {'patron': 143}
niveles: {'titulo': 10, 'articulo': 111, 'capitulo': 18, 'disposiciones': 4} · hojas: 111
avisos: ["Descartado por tipografía (no está en negrita): 'Art. 317.- Lavado de activos. - Será sancionado por lavado d' (pág. 36).", "Descartado por tipografía (no está en negrita): 'Art. 366.- Terrorismo. - La persona que individualmente o fo' (pág. 38).", 'Seccionado incierto en Artículo 1 (pág. 7): cuelga del mismo padre que una sección de nivel superior.']



esperados (regex independiente): 99 · rango 1–99 · sin saltos ni repetidos: True
casos dudosos revisados a mano:
   Art. 317.- Lavado de activos. - Será sancionado por lavado de activos la persona que de
   Art. 366.- Terrorismo. - La persona que individualmente o formando asociaciones armadas,
   Art. 367.- Financiación del terrorismo. - La persona que en forma individual o colectiva,
   Art. 552.- Órdenes especiales en materia de inmovilización preventiva relativa a las
   Art. 553.- Acceso a fondos o levantamiento de la inmovilización preventiva relativa a las
   artículo anterior deberán incluir la siguiente información en la declaración patrimonial


**Revisión manual de los dudosos:** las líneas `Art. 317.-`, `Art. 366.-`, `Art. 367.-`, `Art. 552.-` y `Art. 553.-` son artículos del Código Orgánico Integral Penal que la *Disposición Reformatoria Primera* transcribe: no son artículos de esta ley (se excluyen del conteo y deben quedar dentro del texto de esa disposición). El resto de líneas "Art…" que aparezcan arriba son continuaciones de texto.

In [6]:
arts = [x for x in secciones if x.nivel == "articulo" and re.match(r"art", x.identificador, re.I)]
detectados = [int(re.search(r"\d+", x.identificador).group()) for x in arts]
cd, ce = Counter(detectados), Counter(esperados)
coinciden = sum(min(cd[n], ce[n]) for n in ce)
falsos_pos = sorted((cd - ce).elements())
falsos_neg = sorted((ce - cd).elements())
pct = 100 * coinciden / len(esperados)
print(f"detectados: {len(detectados)} · esperados: {len(esperados)} · coinciden: {coinciden} ({pct:.1f} %)")
print(f"falsos positivos: {len(falsos_pos)} {falsos_pos} · falsos negativos: {len(falsos_neg)} {falsos_neg}")

# Qué patrones se activaron y qué no: nivel de las secciones y primeras líneas de los bloques que debían ser artículo y no lo fueron
print("secciones por nivel:", dict(Counter(x.nivel for x in secciones)))
if falsos_neg:
    for n in falsos_neg:
        previos = [b["texto"][:80] for b in bloques if re.search(rf"Art[ií]culo\s+{n}\b", b["texto"])]
        print(f"  FN Art. {n}: bloques de Docling que lo mencionan → {previos[:2]}")
else:
    print("patrones que fallaron: ninguno (todos los artículos esperados fueron detectados)")

otros = [x for x in secciones if x.nivel == "articulo" and x not in arts]
print("\nDisposiciones detectadas como hojas:", [(x.ruta[-1] if x.ruta else "", x.identificador) for x in otros])
print("Hojas que no son artículos:", Counter(x.nivel for x in secciones if x.es_hoja and x not in arts))
cita = next(x for x in secciones if x.identificador.startswith("PRIMERA") and x.ruta and "REFORMATORIAS" in x.ruta[-1])
print("\nArt. 317 del COIP dentro de la disposición reformatoria:", "Art. 317" in cita.texto_literal, "· ¿es sección propia?:", any(x.identificador.startswith("Art. 317") for x in secciones))
a1, a99 = next(x for x in arts if x.identificador.endswith(" 1")), next(x for x in arts if x.identificador.endswith(" 99"))
print("\nEjemplos:")
for x in (a1, secciones[len(secciones)//2], a99):
    print(f"  {x.id} · {x.nivel} · hoja={x.es_hoja} · págs {x.pagina_inicio}-{x.pagina_fin} · ruta={x.ruta} · titulo={x.titulo!r} · {x.texto_literal[:60]!r}")
print("  el Art. 99 no arrastra el encabezado siguiente:", "DISPOSICIONES GENERALES" not in a99.texto_literal)

detectados: 99 · esperados: 99 · coinciden: 99 (100.0 %)
falsos positivos: 0 [] · falsos negativos: 0 []
secciones por nivel: {'titulo': 10, 'articulo': 111, 'capitulo': 18, 'disposiciones': 4}
patrones que fallaron: ninguno (todos los artículos esperados fueron detectados)

Disposiciones detectadas como hojas: [('DISPOSICIONES GENERALES', 'PRIMERA'), ('DISPOSICIONES GENERALES', 'SEGUNDA'), ('DISPOSICIONES GENERALES', 'TERCERA'), ('DISPOSICIONES TRANSITORIAS', 'PRIMERA'), ('DISPOSICIONES TRANSITORIAS', 'SEGUNDA'), ('DISPOSICIONES TRANSITORIAS', 'TERCERA'), ('DISPOSICIONES REFORMATORIAS', 'PRIMERA'), ('DISPOSICIONES REFORMATORIAS', 'SEGUNDA'), ('DISPOSICIONES REFORMATORIAS', 'TERCERA'), ('DISPOSICIONES DEROGATORIAS', 'PRIMERA'), ('DISPOSICIONES DEROGATORIAS', 'SEGUNDA'), ('', 'DISPOSICIÓN FINAL')]
Hojas que no son artículos: Counter({'articulo': 12})

Art. 317 del COIP dentro de la disposición reformatoria: True · ¿es sección propia?: False

Ejemplos:
  N1:T-1/ART-1 · articulo · hoja=Tr

## 4b · Jerarquía: ruta de cada artículo contra una referencia independiente
**Referencia ("la verdad"):** regex independiente sobre el texto de PyMuPDF (no usa el seccionador ni Docling): líneas `LIBRO|TÍTULO|CAPÍTULO|SECCIÓN <romano>` en mayúsculas y `Artículo N.-`; la ruta se arma con una pila y se corta en `DISPOSICIONES`. El nivel de un encabezado sale del **tipo de patrón** (Libro > Título > Capítulo > Sección > Artículo), no de la tipografía: TÍTULO y CAPÍTULO se ven idénticos (12 pt, negrita, centrado). La tipografía solo **valida** (negrita, línea corta, abre la línea). Se informa el número tal cual sale.

In [7]:
import unicodedata

def sn(t):  # sin tildes ni signos, para comparar rutas
    return " ".join(re.findall(r"\w+", unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode().casefold()))

def ruta_referencia(pdf):
    """{n° de artículo: [libro, título, capítulo, sección]} según PyMuPDF, con una regex independiente."""
    niveles = ["LIBRO", "TITULO", "CAPITULO", "SECCION"]
    pila, ref = {}, {}
    cab = re.compile(r"^[ \t]*(LIBRO|T[ÍI]TULO|CAP[ÍI]TULO|SECCI[ÓO]N)[ \t]+([IVXLCDM]+)\b")
    art = re.compile(r"^[ \t]*Art[ií]culo[ \t]+(\d+)[ \t]*\.?[ \t]*-", re.I)
    for linea in "\n".join(p.get_text() for p in pymupdf.open(pdf)).splitlines():
        if re.match(r"[ \t]*DISPOSICIONES\b", linea):
            break
        m = cab.match(linea)
        if m:
            clave = sn(m.group(1)).upper()
            for n in niveles[niveles.index(clave):]:
                pila.pop(n, None)
            pila[clave] = sn(f"{m.group(1)} {m.group(2)}")
        elif (m := art.match(linea)):
            ref.setdefault(int(m.group(1)), [pila[n] for n in niveles if n in pila])
    return ref

def informe_jerarquia(nombre, pdf, secs, avs, blqs):
    ref = ruta_referencia(pdf)
    arts_ = [x for x in secs if x.nivel == "articulo" and re.match(r"art", x.identificador, re.I)]
    ok_ = [(x, [sn(r) for r in x.ruta] == ref.get(int(re.search(r"\d+", x.identificador).group()))) for x in arts_]
    correctos = sum(c for _, c in ok_)
    malos_inciertos = sum(1 for x, c in ok_ if not c and x.seccionado_incierto)
    buenos_inciertos = sum(1 for x, c in ok_ if c and x.seccionado_incierto)
    # candidatos que una regex laxa tomaría como encabezado de artículo y que no abren línea en negrita
    neg = negritas_pdf(pdf)
    laxos = [b for b in blqs if re.match(r"[ \t]*art[ií]culo[ \t]+\d+", b["texto"], re.I)]
    laxos_no_neg = [b for b in laxos if not any(sn(b["texto"]).startswith(t) for q in (b["pagina"], b["pagina"] + 1) for t in neg.get(q, ()))]
    descartados = [a for a in avs if a.startswith("Descartado por tipografía")]
    aceptados = [a for a in avs if a.startswith("Encabezado aceptado")]
    print(f"== {nombre}")
    print(f"artículos: {len(arts_)} detectados · {len(ref)} en la referencia · rutas correctas: {correctos}/{len(arts_)} ({100*correctos/max(len(arts_),1):.1f} %)")
    print(f"secciones con seccionado_incierto: {sum(x.seccionado_incierto for x in secs)} de {len(secs)} · artículos con ruta incorrecta marcados inciertos: {malos_inciertos}/{len(arts_)-correctos} · artículos correctos marcados inciertos: {buenos_inciertos}")
    print(f"filtro tipográfico: {len(descartados)} candidatos descartados por no estar en negrita {[d.split(': ', 1)[1][:45] for d in descartados]}")
    print(f"  bloques que empiezan con 'Artículo N' (regex laxa): {len(laxos)}, de ellos sin negrita al inicio de línea: {len(laxos_no_neg)} {[b['texto'][:40] for b in laxos_no_neg][:3]}")
    print(f"encabezados validados aceptados con < 3 apariciones: {len(aceptados)} {[a.split(chr(39))[1] for a in aceptados]}")
    print("avisos de jerarquía:", [a[:120] for a in avs if a.startswith("Seccionado incierto")])
    print("niveles del árbol:", dict(Counter(x.nivel for x in secs)), "· ruta de los 2 primeros artículos:", [x.ruta for x in arts_[:2]], "· referencia:", [ref[k] for k in sorted(ref)[:2]])
    return dict(correctos=correctos, total=len(arts_), inciertas=sum(x.seccionado_incierto for x in secs), malos_inciertos=malos_inciertos,
                descartados=len(descartados), aceptados=len(aceptados), laxos_no_neg=len(laxos_no_neg), ref=len(ref))

jer_laft = informe_jerarquia("LA/FT", LAFT, secciones, avisos, bloques)
bl_cap3 = extraer(CAP3)
av_cap3 = []
sec_cap3 = parsear_bloques(bl_cap3, doc_id="SB", tipo_doc="normativa", pdf=CAP3, avisos=av_cap3)
jer_cap3 = informe_jerarquia("L1-XVI-cap-III", CAP3, sec_cap3, av_cap3, bl_cap3)

== LA/FT
artículos: 99 detectados · 99 en la referencia · rutas correctas: 98/99 (99.0 %)
secciones con seccionado_incierto: 1 de 143 · artículos con ruta incorrecta marcados inciertos: 1/1 · artículos correctos marcados inciertos: 0
filtro tipográfico: 2 candidatos descartados por no estar en negrita ["'Art. 317.- Lavado de activos. - Será sancion", "'Art. 366.- Terrorismo. - La persona que indi"]
  bloques que empiezan con 'Artículo N' (regex laxa): 44, de ellos sin negrita al inicio de línea: 0 []
encabezados validados aceptados con < 3 apariciones: 0 []
avisos de jerarquía: ['Seccionado incierto en Artículo 1 (pág. 7): cuelga del mismo padre que una sección de nivel superior.']
niveles del árbol: {'titulo': 10, 'articulo': 111, 'capitulo': 18, 'disposiciones': 4} · ruta de los 2 primeros artículos: [['TÍTULO I'], ['TÍTULO II', 'CAPÍTULO I']] · referencia: [['titulo i', 'capitulo i'], ['titulo ii', 'capitulo i']]


== L1-XVI-cap-III
artículos: 8 detectados · 8 en la referencia · rutas correctas: 8/8 (100.0 %)
secciones con seccionado_incierto: 0 de 13 · artículos con ruta incorrecta marcados inciertos: 0/0 · artículos correctos marcados inciertos: 0
filtro tipográfico: 0 candidatos descartados por no estar en negrita []
  bloques que empiezan con 'Artículo N' (regex laxa): 8, de ellos sin negrita al inicio de línea: 0 []
encabezados validados aceptados con < 3 apariciones: 5 ['LIBRO I', 'TÍTULO XVI', 'CAPÍTULO III', 'SECCIÓN I', 'SECCIÓN II']
avisos de jerarquía: []
niveles del árbol: {'libro': 1, 'titulo': 1, 'capitulo': 1, 'seccion': 2, 'articulo': 8} · ruta de los 2 primeros artículos: [['LIBRO I', 'TÍTULO XVI', 'CAPÍTULO III', 'SECCIÓN I'], ['LIBRO I', 'TÍTULO XVI', 'CAPÍTULO III', 'SECCIÓN I']] · referencia: [['libro i', 'titulo xvi', 'capitulo iii', 'seccion i'], ['libro i', 'titulo xvi', 'capitulo iii', 'seccion i']]


**Lectura:** en la LA/FT, Docling emite el Artículo 1 *antes* de su encabezado `CAPÍTULO I` (orden de lectura): el árbol lo detecta (hijos de niveles distintos bajo un mismo padre) y lo marca incierto en vez de inventar la ruta. En `L1-XVI-cap-III` LIBRO, TÍTULO y CAPÍTULO aparecen una vez y SECCIÓN dos: no llegan a 3 apariciones, pero están **validados** (patrón + negrita + línea propia) y el árbol es consistente, así que se aceptan y las rutas salen completas.


## 4c · Esquemas de encabezado soportados (`patterns.py`)
Cada caso es un texto corto de **entrada** (≥ 3 apariciones, sin PDF: sin validación tipográfica) para el seccionador determinista; se compara el resultado con el esperado. Los esquemas: romanos, arábigos y jerárquicos, letras, ordinales en palabras, `Capítulo 1` / `Sección 2` / `Capítulo Primero`, y `Artículo 5` / `Art. 5` / `5-A` / `5 bis`.

In [8]:
def B(texto, pagina=1):
    return [{"texto": x, "pagina": pagina, "tipo": "parrafo"} for x in texto.split("\n")]

jerarquia = "\n".join([
    "LIBRO I", "TÍTULO I", "CAPÍTULO I", "SECCIÓN I", "Artículo 1.- a", "SECCIÓN II", "Artículo 2.- b", "SECCIÓN III", "Artículo 3.- c",
    "CAPÍTULO II", "Artículo 4.- d", "CAPÍTULO III", "Artículo 5.- e", "TÍTULO II", "CAPÍTULO I", "Artículo 6.- f", "CAPÍTULO II", "Artículo 7.- g",
    "LIBRO II", "TÍTULO III", "CAPÍTULO I", "Artículo 8.- h", "LIBRO III", "TÍTULO IV", "CAPÍTULO I", "Artículo 9.- i"])
ESQUEMAS = {   # nombre: (tipo_doc, texto, identificadores esperados de todas las secciones)
    "romanos (I., II.)": ("manual_control", "I. INTRODUCCIÓN\nt\nII. OBJETIVO\nt\nIII. ALCANCE\nt\nIV. CIERRE\nt", ["I", "II", "III", "IV"]),
    "arábigos y jerárquicos (1., 1.1, 2.1.1)": ("manual_control", "1. Introducción\nt\n1.1 Alcance\nt\n1.2 Objeto\nt\n2. Controles\nt\n2.1 Accesos\nt\n2.1.1 Claves\nt",
                                              ["1", "1.1", "1.2", "2", "2.1", "2.1.1"]),
    "letras (A., B.)": ("manual_control", "A. Objetivo\nt\nB. Alcance\nt\nC. Responsables\nt\nD. Vigencia\nt", ["A", "B", "C", "D"]),
    "ordinales en palabras (PRIMERO … DÉCIMO PRIMERO)": ("normativa", "PRIMERO.- Uno\nt\nSegundo.- Dos\nt\nTERCERO.- Tres\nt\nDÉCIMO.- Diez\nt\nDÉCIMO PRIMERO.- Once\nt",
                                                       ["PRIMERO", "Segundo", "TERCERO", "DÉCIMO", "DÉCIMO PRIMERO"]),
    "Capítulo 1 / Sección 2 (arábigos)": ("normativa", "CAPÍTULO 1\nSección 1\nArtículo 1.- a\nSección 2\nArtículo 2.- b\nSección 3\nArtículo 3.- c\nCAPÍTULO 2\nArtículo 4.- d\nCAPÍTULO 3\nArtículo 5.- e",
                                         ["CAPÍTULO 1", "Sección 1", "Artículo 1", "Sección 2", "Artículo 2", "Sección 3", "Artículo 3", "CAPÍTULO 2", "Artículo 4", "CAPÍTULO 3", "Artículo 5"]),
    "Capítulo Primero / Décimo Primero (palabras)": ("normativa", "CAPÍTULO PRIMERO\nArtículo 1.- a\nCAPÍTULO SEGUNDO\nArtículo 2.- b\nCAPÍTULO TERCERO\nArtículo 3.- c\nCAPÍTULO DÉCIMO PRIMERO\nArtículo 4.- d",
                                                    ["CAPÍTULO PRIMERO", "Artículo 1", "CAPÍTULO SEGUNDO", "Artículo 2", "CAPÍTULO TERCERO", "Artículo 3", "CAPÍTULO DÉCIMO PRIMERO", "Artículo 4"]),
    "Artículo 5 / Art. 6 / 6-A / 6 bis": ("normativa", "Artículo 5.- a\nArt. 6.- b\nArtículo 6-A.- c\nArt. 6 bis.- d\nArtículo 7.- e",
                                         ["Artículo 5", "Art. 6", "Artículo 6-A", "Art. 6 bis", "Artículo 7"]),
}
resultado_esquemas = {}
for nombre, (tipo, texto, esperado) in ESQUEMAS.items():
    avs_e = []
    secs_e = parsear_bloques(B(texto), "D", tipo, avisos=avs_e)
    resultado_esquemas[nombre] = [x.identificador for x in secs_e] == esperado and not any(x.seccionado_incierto for x in secs_e)
    print(("✅" if resultado_esquemas[nombre] else "❌"), nombre, "→", [(x.identificador, x.nivel, "hoja" if x.es_hoja else "padre") for x in secs_e])

# Jerarquía completa libro > título > capítulo > sección > artículo, con numeración que se reinicia al abrir cada padre
secs_j = parsear_bloques(B(jerarquia), "D", "normativa")
rutas_j = {x.identificador: x.ruta for x in secs_j if x.nivel == "articulo"}
esperadas_j = {"Artículo 3": ["LIBRO I", "TÍTULO I", "CAPÍTULO I", "SECCIÓN III"], "Artículo 5": ["LIBRO I", "TÍTULO I", "CAPÍTULO III"],
               "Artículo 7": ["LIBRO I", "TÍTULO II", "CAPÍTULO II"], "Artículo 8": ["LIBRO II", "TÍTULO III", "CAPÍTULO I"], "Artículo 9": ["LIBRO III", "TÍTULO IV", "CAPÍTULO I"]}
resultado_esquemas["libro > título > capítulo > sección > artículo"] = all(rutas_j[k] == v for k, v in esperadas_j.items()) and not any(x.seccionado_incierto for x in secs_j)
print("✅" if resultado_esquemas["libro > título > capítulo > sección > artículo"] else "❌", "libro > título > capítulo > sección > artículo →", {k: rutas_j[k] for k in esperadas_j})

✅ romanos (I., II.) → [('I', 'seccion', 'hoja'), ('II', 'seccion', 'hoja'), ('III', 'seccion', 'hoja'), ('IV', 'seccion', 'hoja')]
✅ arábigos y jerárquicos (1., 1.1, 2.1.1) → [('1', 'numeral', 'padre'), ('1.1', 'numeral', 'hoja'), ('1.2', 'numeral', 'hoja'), ('2', 'numeral', 'padre'), ('2.1', 'numeral', 'padre'), ('2.1.1', 'numeral', 'hoja')]
✅ letras (A., B.) → [('A', 'seccion', 'hoja'), ('B', 'seccion', 'hoja'), ('C', 'seccion', 'hoja'), ('D', 'seccion', 'hoja')]
✅ ordinales en palabras (PRIMERO … DÉCIMO PRIMERO) → [('PRIMERO', 'articulo', 'hoja'), ('Segundo', 'articulo', 'hoja'), ('TERCERO', 'articulo', 'hoja'), ('DÉCIMO', 'articulo', 'hoja'), ('DÉCIMO PRIMERO', 'articulo', 'hoja')]
✅ Capítulo 1 / Sección 2 (arábigos) → [('CAPÍTULO 1', 'capitulo', 'padre'), ('Sección 1', 'seccion', 'padre'), ('Artículo 1', 'articulo', 'hoja'), ('Sección 2', 'seccion', 'padre'), ('Artículo 2', 'articulo', 'hoja'), ('Sección 3', 'seccion', 'padre'), ('Artículo 3', 'articulo', 'hoja'), ('CAPÍTULO 2', '

## 5 · Los 3 manuales MOCK (hojas = numerales)
Docling real sobre cada PDF; `tipo_doc="manual_control"`. Se comparan los numerales detectados (`4.1`, `5.1`…) con los del texto de PyMuPDF, excluyendo la página del índice.

In [9]:
resumen_mocks = {}
for pdf in MOCKS:
    t0 = time.perf_counter()
    bl = extraer(pdf)
    t_ext = time.perf_counter() - t0
    avs = []
    sec = parsear_bloques(bl, doc_id=pdf.stem, tipo_doc="manual_control", pdf=pdf, avisos=avs)
    hojas = [x for x in sec if x.es_hoja]
    numerales = [x for x in hojas if x.nivel == "numeral"]
    # Esperados: líneas "N.N  Título" fuera del índice (cada numeral aparece una vez en el índice y una en el cuerpo)
    cuenta = Counter(m for p in pymupdf.open(pdf) for m in re.findall(r"(?m)^(\d+\.\d+)[ \t]+[A-ZÁÉÍÓÚ]", p.get_text()))
    esperados_n = sorted(k for k, v in cuenta.items() if v >= 2)
    det_n = sorted(x.identificador for x in numerales)
    # Regla de docs/07: un patrón con < 3 apariciones no es "del documento"; sus líneas quedan como texto del padre.
    resumen_mocks[pdf.stem] = dict(extraccion_s=round(t_ext, 1), secciones=len(sec), hojas=len(hojas), numerales=len(numerales),
                                   esperados=len(esperados_n), ok=det_n == (esperados_n if len(esperados_n) >= 3 else []), estrategia=sorted({x.estrategia for x in sec}), inciertas=sum(x.seccionado_incierto for x in sec), avisos=avs)
    print(pdf.stem, resumen_mocks[pdf.stem])
    print("   hojas:", [(x.identificador, x.nivel) for x in hojas])
    print("   ejemplo:", sec[len(sec)//2].id, "ruta", sec[len(sec)//2].ruta, repr(sec[len(sec)//2].texto_literal[:60]))

MOCK-DEMO-01 {'extraccion_s': 40.3, 'secciones': 21, 'hojas': 19, 'numerales': 8, 'esperados': 8, 'ok': True, 'estrategia': ['patron'], 'inciertas': 0, 'avisos': []}
   hojas: [('I', 'seccion'), ('II', 'seccion'), ('III', 'seccion'), ('4.1', 'numeral'), ('5.1', 'numeral'), ('5.2', 'numeral'), ('5.3', 'numeral'), ('5.4', 'numeral'), ('5.5', 'numeral'), ('5.6', 'numeral'), ('5.7', 'numeral'), ('VI', 'seccion'), ('VII', 'seccion'), ('VIII', 'seccion'), ('IX', 'seccion'), ('X', 'seccion'), ('XI', 'seccion'), ('XII', 'seccion'), ('XIII', 'seccion')]
   ejemplo: MOCK-DEMO-01:R-5/N-5.5 ruta ['V'] '5.5 Listas de control\nLa entidad realiza consultas en listas'


MOCK-DEMO-02 {'extraccion_s': 35.7, 'secciones': 12, 'hojas': 12, 'numerales': 0, 'esperados': 2, 'ok': True, 'estrategia': ['patron'], 'inciertas': 0, 'avisos': []}
   hojas: [('I', 'seccion'), ('II', 'seccion'), ('III', 'seccion'), ('IV', 'seccion'), ('V', 'seccion'), ('VI', 'seccion'), ('VII', 'seccion'), ('VIII', 'seccion'), ('IX', 'seccion'), ('X', 'seccion'), ('XI', 'seccion'), ('XII', 'seccion')]
   ejemplo: MOCK-DEMO-02:R-7 ruta [] 'VII. GESTIÓN DOCUMENTAL\nLos documentos generados en la opera'


MOCK-DEMO-03 {'extraccion_s': 49.1, 'secciones': 24, 'hojas': 22, 'numerales': 11, 'esperados': 11, 'ok': True, 'estrategia': ['patron'], 'inciertas': 0, 'avisos': []}
   hojas: [('I', 'seccion'), ('II', 'seccion'), ('III', 'seccion'), ('4.1', 'numeral'), ('5.1', 'numeral'), ('5.2', 'numeral'), ('5.3', 'numeral'), ('5.4', 'numeral'), ('5.5', 'numeral'), ('5.6', 'numeral'), ('5.7', 'numeral'), ('5.8', 'numeral'), ('5.9', 'numeral'), ('5.10', 'numeral'), ('VI', 'seccion'), ('VII', 'seccion'), ('VIII', 'seccion'), ('IX', 'seccion'), ('X', 'seccion'), ('XI', 'seccion'), ('XII', 'seccion'), ('XIII', 'seccion')]
   ejemplo: MOCK-DEMO-03:R-5/N-5.7 ruta ['V'] '5.7 Debida diligencia continua y monitoreo transaccional\nLa '


## 6 · Niveles 2 y 3 de la cascada con Docling real
Dos PDFs generados con PyMuPDF: uno con encabezados **sin numeración** (solo tamaño y negrita → tipografía) y otro de texto corrido sin estructura (→ longitud, `seccionado_incierto=True`).

In [10]:
carpeta = Path(tempfile.mkdtemp())

def crear_pdf(ruta, partes):
    doc = pymupdf.open()
    pagina, y = doc.new_page(), 60
    for fuente, tam, texto in partes:
        if y > 760:
            pagina, y = doc.new_page(), 60
        pagina.insert_textbox(pymupdf.Rect(50, y, 545, y + 140), texto, fontname=fuente, fontsize=tam)
        y += 22 if tam > 12 else 70
    doc.save(ruta)

lorem = "La entidad mantiene registros actualizados de sus operaciones y los conserva para consulta de la autoridad competente. " * 3
partes = []
for tema in ["Politica general", "Control de accesos", "Gestion de incidentes", "Respaldo de informacion", "Capacitacion del personal"]:
    partes += [("hebo", 16, tema), ("helv", 10, lorem)]
con_encabezados = carpeta / "tipografia.pdf"; crear_pdf(con_encabezados, partes)
plano = carpeta / "plano.pdf"; crear_pdf(plano, [("helv", 10, lorem)] * 14)

res_niveles = {}
for nombre, pdf in (("tipografia", con_encabezados), ("longitud", plano)):
    bl = extraer(pdf)
    sec = parsear_bloques(bl, doc_id=nombre.upper(), tipo_doc="manual_control", pdf=pdf)
    res_niveles[nombre] = sec
    print(nombre, "→", len(bl), "bloques,", len(sec), "secciones · estrategia", {x.estrategia for x in sec}, "· inciertas", {x.seccionado_incierto for x in sec}, "· hojas", sum(x.es_hoja for x in sec))
    for x in sec[:3]:
        print("   ", x.id, x.nivel, len(x.texto_literal), "car.", repr(x.texto_literal[:50]))

tipografia → 10 bloques, 5 secciones · estrategia {'tipografia'} · inciertas {False} · hojas 5
    TIPOGRAFIA:POLITICA-GENERAL seccion 373 car. 'Politica general\nLa entidad mantiene registros act'
    TIPOGRAFIA:CONTROL-DE-ACCESOS seccion 375 car. 'Control de accesos\nLa entidad mantiene registros a'
    TIPOGRAFIA:GESTION-DE-INCIDENTES seccion 378 car. 'Gestion de incidentes\nLa entidad mantiene registro'


longitud → 2 bloques, 2 secciones · estrategia {'longitud'} · inciertas {True} · hojas 2
    LONGITUD:BLOQUE-1 bloque 3926 car. 'La entidad mantiene registros actualizados de sus '
    LONGITUD:BLOQUE-2 bloque 1070 car. 'La entidad mantiene registros actualizados de sus '


## 7 · Regex sin costo cuadrático y límite de bloque (appsec)
Antes: `^\s*` con `MULTILINE` sobre el bloque entero (catálogo de docs/07). Ahora: `[ \t]*` y solo la primera línea (acotada).

In [11]:
import re as _re
antes = _re.compile(r"^\s*(ART[IÍ]CULO|ART\.?)\s*(\d+(\.\d+)*|[IVXLCDM]+|[A-Z]+)(\s*\.?-|\.|:|\s)", _re.I | _re.M)
hostil_8k = "\n" * 8000 + "x"
t0 = time.perf_counter(); antes.search(hostil_8k); t_antes = time.perf_counter() - t0
print(f"regex del catálogo original, 8.000 saltos de línea: {t_antes:.2f} s")

hostiles = {
    "100 KB de saltos y espacios": ("\n \t" * 25000)[:100_000],
    "100 KB de espacios": " " * 100_000,
    "100 KB, 8.000 líneas y 'Artículo' al final": ("\n" * 8000 + " " * 92_000 + "Artículo")[:100_000],
    "100 KB 'Artículo 1111…'": "Artículo " + "1" * 99_900,
}
for nombre_h, h in {"letra 'A.' + 100 KB de espacios": ". A." + " " * 100_000, "ordinal compuesto + 100 KB de espacios": ". DÉCIMO" + " " * 100_000 + "PRIMERO",
                    "CAPÍTULO DÉCIMO + 100 KB de espacios": "CAPÍTULO DÉCIMO" + " " * 100_000, "Artículo 5 + 100 KB de guiones": ". Artículo 5" + "-" * 100_000,
                    "Artículo 5 + espacios + 'bis'": ". Artículo 5" + " " * 50_000 + "bis" + " " * 50_000, "romano 'I.' + 100 KB de espacios": "I." + " " * 100_000,
                    "ordinal + 100 KB de tabuladores": "\nPRIMERO" + "\t" * 100_000}.items():
    hostiles[nombre_h] = h[:100_000]
tiempos = {}
for pre in ("Artículo 1", "PRIMERA", "\nArtículo 1"):          # ReDoS en el corte de artículos (appsec)
    for nombre_ch, ch in (("espacios", " "), ("tabuladores", "\t")):
        hostiles[f"{pre!r} + 100 KB de {nombre_ch}"] = (". " + pre + ch * 100_000)[:100_000]
for nombre, h in hostiles.items():
    assert len(h) <= 100_000
    t0 = time.perf_counter()
    parsear_bloques([{"texto": h, "pagina": 1, "tipo": "parrafo"}], "H", "normativa")
    tiempos[nombre] = time.perf_counter() - t0
    print(f"{nombre}: {tiempos[nombre]*1000:.2f} ms")
try:
    parsear_bloques([{"texto": "a" * (s.SECCION_BLOQUE_MAX + 1), "pagina": 1, "tipo": "parrafo"}], "H", "normativa")
    limite = None
except SeccionadoError as e:
    limite = e.codigo
    print("bloque sobre el límite →", e.codigo, "·", e.para_usuario()["mensaje"])

regex del catálogo original, 8.000 saltos de línea: 0.56 s
100 KB de saltos y espacios: 23.42 ms
100 KB de espacios: 3.78 ms
100 KB, 8.000 líneas y 'Artículo' al final: 29.64 ms
100 KB 'Artículo 1111…': 3.83 ms
letra 'A.' + 100 KB de espacios: 29.59 ms
ordinal compuesto + 100 KB de espacios: 20.45 ms
CAPÍTULO DÉCIMO + 100 KB de espacios: 4.07 ms


Artículo 5 + 100 KB de guiones: 4.22 ms
Artículo 5 + espacios + 'bis': 7.69 ms
romano 'I.' + 100 KB de espacios: 29.29 ms
ordinal + 100 KB de tabuladores: 3.43 ms
'Artículo 1' + 100 KB de espacios: 11.42 ms
'Artículo 1' + 100 KB de tabuladores: 11.71 ms
'PRIMERA' + 100 KB de espacios: 3.62 ms
'PRIMERA' + 100 KB de tabuladores: 3.66 ms
'\nArtículo 1' + 100 KB de espacios: 11.17 ms
'\nArtículo 1' + 100 KB de tabuladores: 11.65 ms
bloque sobre el límite → ERR-SEC-001 · El documento tiene un bloque de texto demasiado grande para procesarlo. Revisa que sea el archivo correcto.


## 8 · Lista de chequeo del criterio de aceptación

In [12]:
checks = {
    "Docling real: bloques con tipos de docs/07 (encabezado, parrafo, tabla, lista)": set(tipos) <= {"encabezado", "parrafo", "tabla", "lista"} and len(tipos) >= 3,
    "encabezados con nivel": bool(niveles),
    "el progreso llega al total de páginas": bool(avance) and avance[-1][0] == avance[-1][1],
    "LA/FT: artículos detectados = esperados (≥ 95 %)": pct >= 95,
    "LA/FT: sin falsos positivos ni negativos": not falsos_pos and not falsos_neg,
    "LA/FT: Art. del COIP citados no son secciones": not any(x.identificador.startswith("Art. 3") or x.identificador.startswith("Art. 5") for x in secciones),
    "reprocesar el mismo PDF usa la caché (< 1 s y mismo resultado)": otra_vez == bloques and t_cache < 1,
    "un PDF no válido se rechaza antes de Docling": rechazo is not None,
    "MOCK: numerales detectados = esperados en los 3 manuales": all(v["ok"] for v in resumen_mocks.values()),
    "nivel 2 (tipografía) produce secciones sin numeración": {x.estrategia for x in res_niveles["tipografia"]} == {"tipografia"} and len(res_niveles["tipografia"]) >= 3,
    "nivel 3 (longitud) marca seccionado_incierto": all(x.seccionado_incierto and x.estrategia == "longitud" for x in res_niveles["longitud"]),
    "bloque hostil de 100 KB < 0,1 s (incluye espacios y tabuladores tras Artículo/PRIMERA)": max(tiempos.values()) < 0.1,
    "esquemas: romanos, arábigos, letras, ordinales, Capítulo/Sección numéricos y en palabras, Art. 5-A/bis y jerarquía completa": all(resultado_esquemas.values()),
    "L1-XVI-cap-III: rutas de los 8 artículos correctas con encabezados validados (8/8)": jer_cap3["correctos"] == jer_cap3["total"] == 8,
    "LA/FT: rutas no empeoran (≥ 98/99)": jer_laft["correctos"] >= 98,
    "LA/FT: toda ruta incorrecta queda marcada incierta (no se inventa la ruta)": jer_laft["malos_inciertos"] == jer_laft["total"] - jer_laft["correctos"],
    "bloque sobre el límite → ERR-SEC-001": limite == "ERR-SEC-001",
}
for k, v in checks.items():
    print(("✅" if v else "❌"), k)
assert all(checks.values())
print(f"\nLA/FT: {t_extraccion:.1f} s de extracción, RAM pico {ram_pico:.0f} MB, {len(arts)}/{len(esperados)} artículos.")
print("Jerarquía:", jer_laft, jer_cap3)

✅ Docling real: bloques con tipos de docs/07 (encabezado, parrafo, tabla, lista)
✅ encabezados con nivel
✅ el progreso llega al total de páginas
✅ LA/FT: artículos detectados = esperados (≥ 95 %)
✅ LA/FT: sin falsos positivos ni negativos
✅ LA/FT: Art. del COIP citados no son secciones
✅ reprocesar el mismo PDF usa la caché (< 1 s y mismo resultado)
✅ un PDF no válido se rechaza antes de Docling
✅ MOCK: numerales detectados = esperados en los 3 manuales
✅ nivel 2 (tipografía) produce secciones sin numeración
✅ nivel 3 (longitud) marca seccionado_incierto
✅ bloque hostil de 100 KB < 0,1 s (incluye espacios y tabuladores tras Artículo/PRIMERA)
✅ esquemas: romanos, arábigos, letras, ordinales, Capítulo/Sección numéricos y en palabras, Art. 5-A/bis y jerarquía completa
✅ L1-XVI-cap-III: rutas de los 8 artículos correctas con encabezados validados (8/8)
✅ LA/FT: rutas no empeoran (≥ 98/99)
✅ LA/FT: toda ruta incorrecta queda marcada incierta (no se inventa la ruta)
✅ bloque sobre el límite 